# **Sepsis Early Prediction** (6-Hour Ahead)

Time-series modeling for early sepsis detection using ICU physiological data.
Patient-wise split and structured feature engineering experiments.

Baseline model: XGBoost  
Target: FutureSepsis (6-hour ahead prediction)

## 1. Data Loading

In [11]:
import pandas as pd

baseline_df = pd.read_csv(
    "/content/drive/MyDrive/physionet/baseline_dataset.psv",
    sep="|"
)

print("Shape:", baseline_df.shape)
print("Patients:", baseline_df["PatientID"].nunique())
print("Positive rate (%):", baseline_df["FutureSepsis"].mean() * 100)
baseline_df.head()

Shape: (790215, 14)
Patients: 20336
Positive rate (%): 1.1255164733648437


,HR,O2Sat,SBP,MAP,Resp,Temp,Lactate,WBC,Creatinine,Platelets,Age,ICULOS,FutureSepsis,PatientID
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.14,1,0,p000001
1,97.0,95.0,98.0,75.33,19.0,NaN,NaN,NaN,NaN,NaN,83.14,2,0,p000001
2,89.0,99.0,122.0,86.00,22.0,NaN,NaN,NaN,NaN,NaN,83.14,3,0,p000001
3,90.0,95.0,NaN,NaN,30.0,NaN,NaN,NaN,NaN,NaN,83.14,4,0,p000001
4,103.0,88.5,122.0,91.33,24.5,NaN,NaN,NaN,NaN,NaN,83.14,5,0,p000001


## 2. Train/Test Split (Patient-wise)

In [12]:
from sklearn.model_selection import train_test_split

patients = baseline_df['PatientID'].unique()

train_patients, test_patients = train_test_split(
    patients,
    test_size=0.2,
    random_state=42
)

train_df = baseline_df[baseline_df['PatientID'].isin(train_patients)].copy()
test_df  = baseline_df[baseline_df['PatientID'].isin(test_patients)].copy()

print("Train patients:", train_df['PatientID'].nunique())
print("Test patients:", test_df['PatientID'].nunique())

Train patients: 16268
Test patients: 4068


## 3. Missing Handling Strategy
3.1 Add Missing Indicators (Labs)

In [13]:
lab_cols = ['Lactate','WBC','Creatinine','Platelets']

for col in lab_cols:
    train_df[col + '_missing'] = train_df[col].isnull().astype(int)
    test_df[col + '_missing']  = test_df[col].isnull().astype(int)

3.2 Forward Fill Vitals

In [14]:
vitals = ['HR','O2Sat','SBP','MAP','Resp','Temp']

train_df[vitals] = train_df.groupby('PatientID')[vitals].ffill()
test_df[vitals]  = test_df.groupby('PatientID')[vitals].ffill()

3.3 Median Fill (Train Statistics Only)

In [15]:
vital_medians = train_df[vitals].median()

train_df[vitals] = train_df[vitals].fillna(vital_medians)
test_df[vitals]  = test_df[vitals].fillna(vital_medians)

## 4. Baseline v1 – Raw Features

In [16]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

X_train = train_df.drop(columns=['FutureSepsis','PatientID'])
y_train = train_df['FutureSepsis']

X_test  = test_df.drop(columns=['FutureSepsis','PatientID'])
y_test  = test_df['FutureSepsis']

pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = neg / pos

model_v1 = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
    random_state=42,
    eval_metric='logloss'
)

model_v1.fit(X_train, y_train)

y_pred_v1 = model_v1.predict_proba(X_test)[:, 1]

roc_v1 = roc_auc_score(y_test, y_pred_v1)
pr_v1  = average_precision_score(y_test, y_pred_v1)

print("ROC-AUC v1:", roc_v1)
print("PR-AUC v1:", pr_v1)

ROC-AUC v1: 0.712890249201721
PR-AUC v1: 0.03734182979224381


## 5. Baseline v2 – Rolling Mean Features

In [17]:
for col in vitals:
    train_df[col + '_roll6_mean'] = (
        train_df.groupby('PatientID')[col]
        .rolling(window=6, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

for col in vitals:
    test_df[col + '_roll6_mean'] = (
        test_df.groupby('PatientID')[col]
        .rolling(window=6, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

X_train_v2 = train_df.drop(columns=['FutureSepsis','PatientID'])
X_test_v2  = test_df.drop(columns=['FutureSepsis','PatientID'])

model_v2 = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
    random_state=42,
    eval_metric='logloss'
)

model_v2.fit(X_train_v2, y_train)

y_pred_v2 = model_v2.predict_proba(X_test_v2)[:, 1]

roc_v2 = roc_auc_score(y_test, y_pred_v2)
pr_v2  = average_precision_score(y_test, y_pred_v2)

print("ROC-AUC v2:", roc_v2)
print("PR-AUC v2:", pr_v2)

ROC-AUC v2: 0.7058143406281834
PR-AUC v2: 0.039078233883218336


## 6. Baseline v3 – Delta Features

In [18]:
# Remove rolling features
roll_cols = [c for c in train_df.columns if 'roll6' in c]
train_df.drop(columns=roll_cols, inplace=True)
test_df.drop(columns=roll_cols, inplace=True)

# Add delta6
for col in vitals:
    train_df[col + '_delta6'] = (
        train_df.groupby('PatientID')[col].shift(0) -
        train_df.groupby('PatientID')[col].shift(6)
    ).fillna(0)

    test_df[col + '_delta6'] = (
        test_df.groupby('PatientID')[col].shift(0) -
        test_df.groupby('PatientID')[col].shift(6)
    ).fillna(0)

X_train_v3 = train_df.drop(columns=['FutureSepsis','PatientID'])
X_test_v3  = test_df.drop(columns=['FutureSepsis','PatientID'])

model_v3 = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
    random_state=42,
    eval_metric='logloss'
)

model_v3.fit(X_train_v3, y_train)

y_pred_v3 = model_v3.predict_proba(X_test_v3)[:, 1]

roc_v3 = roc_auc_score(y_test, y_pred_v3)
pr_v3  = average_precision_score(y_test, y_pred_v3)

print("ROC-AUC v3:", roc_v3)
print("PR-AUC v3:", pr_v3)

ROC-AUC v3: 0.7223357992130823
PR-AUC v3: 0.03851131244049388


## 7. Results Comparison

In [19]:
import pandas as pd

results = pd.DataFrame({
    "Version": ["v1 - Raw", "v2 - Rolling", "v3 - Delta"],
    "ROC-AUC": [roc_v1, roc_v2, roc_v3],
    "PR-AUC":  [pr_v1, pr_v2, pr_v3]
})

results

,Version,ROC-AUC,PR-AUC
0,v1 - Raw,0.712890,0.037342
1,v2 - Rolling,0.705814,0.039078
2,v3 - Delta,0.722336,0.038511


## 8. **Conclusion**

## Conclusion

- Raw baseline achieved ROC ≈ 0.71.
- Rolling mean features did not improve overall ranking.
- Delta (6-hour change) improved performance to ROC ≈ 0.72.

This suggests that directional physiological change is more informative
than smoothed averages for early sepsis prediction.

Future improvements:
- 1-hour delta (short-term momentum)
- Volatility features
- Logistic regression comparison
- Hyperparameter tuning